# Alpha-identity regularization — both regimes


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 320)


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "opinion_dynamics").exists():
            return candidate
    raise RuntimeError("Could not find repository root.")


REPO_ROOT = find_repo_root()
RESULTS_ROOT = REPO_ROOT / "opinion_dynamics" / "experiments" / "results"

SS_DIR = RESULTS_ROOT / "experiment_2026_09_30_single_shot_alpha_identity_sweep"
AB_DIR = RESULTS_ROOT / "experiment_2026_09_30_abundant_alpha_identity_sweep"
OLD_SS_DIR = RESULTS_ROOT / "experiment_2026_09_29_single_shot_full_regularization_sweep"
OUT_DIR = RESULTS_ROOT / "alpha_identity_sweep_analysis_2026_09_30"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SS_PATH = SS_DIR / "single_shot_alpha_identity_sweep_seed0.csv"
AB_PATH = AB_DIR / "regularization_sweep_all_runs.csv"
OLD_SS_PATH = OLD_SS_DIR / "single_shot_full_regularization_sweep_seed0.csv"

for p in [SS_PATH, AB_PATH, OLD_SS_PATH]:
    if not p.exists():
        raise FileNotFoundError(p)

print("Repo:", REPO_ROOT)


Repo: d:\Work\repos\RL\unknown_graph_networks


C:\Users\Chainsword\AppData\Local\Temp\ipykernel_105948\3427906824.py:3: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:
METRICS = [
    "control", "prediction_ratio", "A_MAE", "A_Fro", "row_TV",
    "true_edge_mass", "support_topdegree_recall", "edge_AP",
    "centrality_L1", "row_entropy_norm", "effective_neighbors",
]

ss = pd.read_csv(SS_PATH)
old_ss = pd.read_csv(OLD_SS_PATH)
ss_none = old_ss[
    (old_ss["regularizer_kind"] == "none")
    & (old_ss["regularizer_lambda"].astype(float) == 0.0)
].copy()
assert len(ss_none) == 195

ss_alpha = ss.copy()
assert set(ss_alpha["regularizer_kind"].unique()) == {"alpha_identity_l2"}


def canon_ss(df):
    out = df.copy()
    out["regime"] = "single_shot"
    out["control"] = out["final_mean"]
    out["prediction_ratio"] = out["final_fit_model_over_identity"]
    return out


ab = pd.read_csv(AB_PATH)
assert set(ab["regularizer_kind"].unique()) == {"none", "alpha_identity_l2"}
ab = ab.copy()
ab["regime"] = "abundant"
ab["control"] = ab["mean_end"]
ab["prediction_ratio"] = ab["model_over_identity"]

work = pd.concat(
    [canon_ss(ss_none), canon_ss(ss_alpha), ab],
    ignore_index=True,
    sort=False,
)

counts = work.groupby(["regime", "regularizer_kind", "regularizer_lambda"]).size()
assert counts.eq(195).all(), counts[counts != 195]
print("Validated all conditions at 195 environments each.")


Validated all conditions at 195 environments each.


In [3]:
ref = (
    work[work["regularizer_kind"] == "none"]
    .set_index(["regime", "environment_id"])
)

for metric in METRICS:
    work[f"delta_{metric}"] = work.apply(
        lambda r: float(r[metric]) - float(
            ref.loc[(r["regime"], r["environment_id"]), metric]
        ),
        axis=1,
    )

summary = (
    work.groupby(["regime", "regularizer_kind", "regularizer_lambda"], as_index=False)
    .agg(
        n=("environment_id", "size"),
        control=("control", "mean"),
        delta_control=("delta_control", "mean"),
        prediction=("prediction_ratio", "mean"),
        delta_prediction=("delta_prediction_ratio", "mean"),
        A_MAE=("A_MAE", "mean"),
        delta_A_MAE=("delta_A_MAE", "mean"),
        row_TV=("row_TV", "mean"),
        delta_row_TV=("delta_row_TV", "mean"),
        true_edge_mass=("true_edge_mass", "mean"),
        delta_true_edge_mass=("delta_true_edge_mass", "mean"),
        edge_AP=("edge_AP", "mean"),
        delta_edge_AP=("delta_edge_AP", "mean"),
        centrality_L1=("centrality_L1", "mean"),
        delta_centrality_L1=("delta_centrality_L1", "mean"),
        effective_neighbors=("effective_neighbors", "mean"),
        delta_effective_neighbors=("delta_effective_neighbors", "mean"),
    )
)

by_dyn = (
    work.groupby(
        ["regime", "regularizer_kind", "regularizer_lambda", "scenario_class", "dynamics"],
        as_index=False,
    )
    .agg(
        n=("environment_id", "size"),
        delta_control=("delta_control", "mean"),
        delta_prediction=("delta_prediction_ratio", "mean"),
        delta_A_MAE=("delta_A_MAE", "mean"),
        delta_row_TV=("delta_row_TV", "mean"),
        delta_true_edge_mass=("delta_true_edge_mass", "mean"),
        delta_edge_AP=("delta_edge_AP", "mean"),
        delta_centrality_L1=("delta_centrality_L1", "mean"),
    )
)

worst = (
    by_dyn.groupby(["regime", "regularizer_kind", "regularizer_lambda"], as_index=False)
    .agg(
        worst_dynamic_control_delta=("delta_control", "min"),
        worst_dynamic_row_TV_delta=("delta_row_TV", "max"),
        worst_dynamic_A_MAE_delta=("delta_A_MAE", "max"),
    )
)
summary = summary.merge(
    worst,
    on=["regime", "regularizer_kind", "regularizer_lambda"],
    how="left",
)

summary.to_csv(OUT_DIR / "alpha_identity_summary.csv", index=False)
by_dyn.to_csv(OUT_DIR / "alpha_identity_by_dynamics.csv", index=False)
display(summary.round(6))


,regime,regularizer_kind,regularizer_lambda,n,control,delta_control,prediction,delta_prediction,A_MAE,delta_A_MAE,row_TV,delta_row_TV,true_edge_mass,delta_true_edge_mass,edge_AP,delta_edge_AP,centrality_L1,delta_centrality_L1,effective_neighbors,delta_effective_neighbors,worst_dynamic_control_delta,worst_dynamic_row_TV_delta,worst_dynamic_A_MAE_delta
0,abundant,alpha_identity_l2,0.000000,195,0.768632,0.000004,0.638296,-0.000014,0.076801,0.000001,0.576007,0.000011,0.433513,-0.000011,0.741841,-0.000007,0.480078,-0.000004,9.154989,0.000162,0.000000,0.000058,0.000008
1,abundant,alpha_identity_l2,0.000000,195,0.768632,0.000004,0.638270,-0.000041,0.076804,0.000004,0.576028,0.000032,0.433491,-0.000032,0.741854,0.000006,0.480070,-0.000011,9.155313,0.000487,0.000000,0.000173,0.000023
2,abundant,alpha_identity_l2,0.000000,195,0.768648,0.000020,0.638181,-0.000130,0.076814,0.000014,0.576102,0.000105,0.433417,-0.000106,0.741807,-0.000041,0.480046,-0.000036,9.156447,0.001621,-0.000051,0.000575,0.000077
3,abundant,alpha_identity_l2,0.000000,195,0.768643,0.000016,0.637969,-0.000341,0.076841,0.000042,0.576311,0.000315,0.433205,-0.000318,0.741613,-0.000235,0.479980,-0.000101,9.159670,0.004843,-0.000070,0.001716,0.000229
4,abundant,alpha_identity_l2,0.000001,195,0.768641,0.000013,0.637665,-0.000645,0.076937,0.000138,0.577030,0.001034,0.432480,-0.001043,0.740975,-0.000872,0.479798,-0.000284,9.170748,0.015922,-0.000154,0.005616,0.000749
5,abundant,alpha_identity_l2,0.000003,195,0.768655,0.000027,0.639326,0.001016,0.077194,0.000395,0.578956,0.002960,0.430544,-0.002979,0.739369,-0.002478,0.479633,-0.000448,9.200635,0.045809,-0.000511,0.015938,0.002125
6,abundant,alpha_identity_l2,0.000010,195,0.768854,0.000227,0.655620,0.017310,0.077913,0.001114,0.584350,0.008353,0.425175,-0.008349,0.734488,-0.007360,0.481257,0.001176,9.286358,0.131531,-0.000486,0.043385,0.005785
7,abundant,alpha_identity_l2,0.000030,195,0.769159,0.000531,0.700400,0.062090,0.079081,0.002281,0.593105,0.017109,0.416537,-0.016986,0.727132,-0.014716,0.487914,0.007833,9.430954,0.276128,-0.000638,0.082519,0.011002
8,abundant,none,0.000000,195,0.768628,0.000000,0.638310,0.000000,0.076800,0.000000,0.575996,0.000000,0.433523,0.000000,0.741848,0.000000,0.480081,0.000000,9.154827,0.000000,0.000000,0.000000,0.000000
9,single_shot,alpha_identity_l2,0.000000,195,0.763101,-0.000019,0.171390,-0.000354,0.051145,0.000017,0.383586,0.000128,0.658149,0.001234,0.704397,-0.000126,0.502506,0.000504,3.864759,-0.013995,-0.001416,0.002342,0.000312


In [4]:
alpha = summary[summary["regularizer_kind"] == "alpha_identity_l2"].copy()
cols = [
    "delta_control", "delta_prediction", "delta_A_MAE", "delta_row_TV",
    "delta_true_edge_mass", "delta_edge_AP", "delta_centrality_L1",
    "delta_effective_neighbors", "worst_dynamic_control_delta",
    "worst_dynamic_row_TV_delta",
]

parts = []
for regime in ["abundant", "single_shot"]:
    p = alpha[alpha["regime"] == regime][["regularizer_lambda"] + cols].copy()
    p = p.rename(columns={c: f"{regime}_{c}" for c in cols})
    parts.append(p)

joint = parts[0].merge(parts[1], on="regularizer_lambda", validate="one_to_one")
joint = joint.sort_values("regularizer_lambda")

joint["row_TV_better_both"] = (
    (joint["abundant_delta_row_TV"] < 0)
    & (joint["single_shot_delta_row_TV"] < 0)
)
joint["A_MAE_better_both"] = (
    (joint["abundant_delta_A_MAE"] < 0)
    & (joint["single_shot_delta_A_MAE"] < 0)
)
joint["control_within_1e3_both"] = (
    (joint["abundant_delta_control"].abs() <= 1e-3)
    & (joint["single_shot_delta_control"].abs() <= 1e-3)
)

joint.to_csv(OUT_DIR / "alpha_identity_side_by_side.csv", index=False)
print("Side by side")
display(joint.round(6))

print("Candidates: row-TV better in both + |control delta| <= 0.001 in both")
display(
    joint[
        joint["row_TV_better_both"]
        & joint["control_within_1e3_both"]
    ].round(6)
)

print("Outputs:", OUT_DIR)
print("ALPHA-IDENTITY ANALYSIS COMPLETE")


Side by side


,regularizer_lambda,abundant_delta_control,abundant_delta_prediction,abundant_delta_A_MAE,abundant_delta_row_TV,abundant_delta_true_edge_mass,abundant_delta_edge_AP,abundant_delta_centrality_L1,abundant_delta_effective_neighbors,abundant_worst_dynamic_control_delta,abundant_worst_dynamic_row_TV_delta,single_shot_delta_control,single_shot_delta_prediction,single_shot_delta_A_MAE,single_shot_delta_row_TV,single_shot_delta_true_edge_mass,single_shot_delta_edge_AP,single_shot_delta_centrality_L1,single_shot_delta_effective_neighbors,single_shot_worst_dynamic_control_delta,single_shot_worst_dynamic_row_TV_delta,row_TV_better_both,A_MAE_better_both,control_within_1e3_both
0,0.000000,0.000004,-0.000014,0.000001,0.000011,-0.000011,-0.000007,-0.000004,0.000162,0.000000,0.000058,-0.000019,-0.000354,0.000017,0.000128,0.001234,-0.000126,0.000504,-0.013995,-0.001416,0.002342,False,False,True
1,0.000000,0.000004,-0.000041,0.000004,0.000032,-0.000032,0.000006,-0.000011,0.000487,0.000000,0.000173,-0.000117,0.000226,0.000224,0.001680,0.002985,-0.001188,0.004435,-0.032370,-0.001178,0.011994,False,False,True
2,0.000000,0.000020,-0.000130,0.000014,0.000105,-0.000106,-0.000041,-0.000036,0.001621,-0.000051,0.000575,0.000209,0.001092,0.000552,0.004139,0.009467,-0.002150,0.009353,-0.083613,-0.002543,0.041460,False,False,True
3,0.000000,0.000016,-0.000341,0.000042,0.000315,-0.000318,-0.000235,-0.000101,0.004843,-0.000070,0.001716,-0.000243,0.007855,0.000755,0.005659,0.015471,-0.010956,0.008866,-0.159662,-0.006023,0.069054,False,False,True
4,0.000001,0.000013,-0.000645,0.000138,0.001034,-0.001043,-0.000872,-0.000284,0.015922,-0.000154,0.005616,-0.001384,0.038552,0.003052,0.022891,0.018804,-0.040255,0.052169,-0.357669,-0.010759,0.089241,False,False,False
5,0.000003,0.000027,0.001016,0.000395,0.002960,-0.002979,-0.002478,-0.000448,0.045809,-0.000511,0.015938,-0.006488,0.076520,0.006424,0.048176,0.002087,-0.072479,0.083312,-0.504278,-0.022660,0.132784,False,False,False
6,0.000010,0.000227,0.017310,0.001114,0.008353,-0.008349,-0.007360,0.001176,0.131531,-0.000486,0.043385,-0.015047,0.129144,0.010940,0.082051,-0.016282,-0.108926,0.177943,-0.838572,-0.043277,0.241088,False,False,False
7,0.000030,0.000531,0.062090,0.002281,0.017109,-0.016986,-0.014716,0.007833,0.276128,-0.000638,0.082519,-0.028019,0.204945,0.016651,0.124885,-0.043246,-0.149227,0.242380,-0.880658,-0.069886,0.337592,False,False,False


Candidates: row-TV better in both + |control delta| <= 0.001 in both


,regularizer_lambda,abundant_delta_control,abundant_delta_prediction,abundant_delta_A_MAE,abundant_delta_row_TV,abundant_delta_true_edge_mass,abundant_delta_edge_AP,abundant_delta_centrality_L1,abundant_delta_effective_neighbors,abundant_worst_dynamic_control_delta,abundant_worst_dynamic_row_TV_delta,single_shot_delta_control,single_shot_delta_prediction,single_shot_delta_A_MAE,single_shot_delta_row_TV,single_shot_delta_true_edge_mass,single_shot_delta_edge_AP,single_shot_delta_centrality_L1,single_shot_delta_effective_neighbors,single_shot_worst_dynamic_control_delta,single_shot_worst_dynamic_row_TV_delta,row_TV_better_both,A_MAE_better_both,control_within_1e3_both


Outputs: d:\Work\repos\RL\unknown_graph_networks\opinion_dynamics\experiments\results\alpha_identity_sweep_analysis_2026_09_30
ALPHA-IDENTITY ANALYSIS COMPLETE
